In [ ]:
from utils.gpu import set_visible_gpu

set_visible_gpu(0)

In [ ]:
import config
from utils.training import enable_reproducibility

enable_reproducibility(config.seed)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import pandas as pd
import numpy as np
import os
import random
from collections import defaultdict

from utils.evaluation import generate_cohort_table

In [ ]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    replacements = {"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"}
    for umlaut, ersatz in replacements.items():
        name = name.replace(umlaut, ersatz)
    return name.replace(" ", "").replace("-", "").lower()

In [ ]:
def label_logic_sbp_met_slope(r):
    if not pd.isna(r["SBP/MET slope2"]):
        if r["SBP/MET slope2"] > 6.2:
            return 1
        else:
            return 0
    elif not pd.isna(r["SBP/MET nach EX"]):
        if r["SBP/MET nach EX"] == 1:
            return 0
        elif r["SBP/MET nach EX"] == 2:
            return 1
        else:
            return np.nan
    else:
        return np.nan

In [ ]:
matching = pd.read_csv(f"{config.directory}/data/{config.matching_csv}")

In [ ]:
# get duplicates for pat id
duplicates = matching[matching.duplicated(subset=["PatID"], keep=False)]

In [ ]:
# get duplicates for name
duplicates = matching[matching.duplicated(subset=["SN"], keep=False)]

In [ ]:
# get duplicates for dob
duplicates = matching[matching.duplicated(subset=["DoB"], keep=False)]

In [ ]:
manualData = pd.read_excel(f"{config.directory}/data/{config.data_excel}", sheet_name=config.sheet_name, nrows=config.num_rows)
manualData["Name_Combined"] = (manualData["Surname"].apply(normalize_name) + manualData["FirstNames"].apply(normalize_name))
manualData["Name_Combined"] = manualData.apply(lambda x: "".join(sorted(x["Name_Combined"])), axis=1)
manualData["DoB_Formatted"] = manualData["DateOfBirth"].dt.strftime("%d.%m.%y")
manualData["MDT_Formatted"] = manualData["MeasurementDateTime"].dt.strftime("%d.%m.%y")
manualData["Label"] = manualData.apply(label_logic_sbp_met_slope, axis=1)
cols = ["Name_Combined"] + ["DoB_Formatted"] + ["Label"] + ["SBP/MET slope2"] + ["RR max sys"] + ["RR max diast."] + ["GenderID"] + [col for col in manualData.columns if col not in["Name_Combined", "DoB_Formatted", "Label", "SBP/MET slope2", "RR max sys", "GenderID", "RR max diast."]]
manualData = manualData[cols]

In [ ]:
# Get Name Duplicates in manualData where date of birth differs
duplicates = manualData[manualData.duplicated(subset=["Name_Combined"], keep=False)].copy()

duplicates_with_different_date = duplicates.groupby("Name_Combined").filter(lambda x: x["DateOfBirth"].nunique() > 1)

In [ ]:
# Get the date of birth duplicates in manualData where the name differs
duplicates = manualData[manualData.duplicated(subset=["DateOfBirth"], keep=False)].copy()

duplicates_with_different_name = duplicates.groupby("DateOfBirth").filter(lambda x: x["Name_Combined"].nunique() > 1)

In [ ]:
# get entries in manualData where no duplicates in DoB occur
non_duplicates = manualData[~manualData.index.isin(duplicates_with_different_name.index)]

In [ ]:
# get the non duplicates in manualData of DoB that occur in matching
matching_dobs = set(matching["DoB"].astype(str))
non_duplicates_in_matching = non_duplicates[non_duplicates["DoB_Formatted"].isin(matching_dobs)]
non_duplicates_not_in_matching = non_duplicates[~non_duplicates["DoB_Formatted"].isin(matching_dobs)]

In [ ]:
# get the entries in manualData with duplicate DoBs that can be matched with sur- and firstnames in matching
duplicates_in_matching = []
duplicates_not_in_matching = []

for _, row in duplicates_with_different_name.iterrows():
    match = matching[
        (matching["DoB"] == row["DoB_Formatted"]) &
        (matching["SN"] == row["Surname"]) &
        (matching["FN"] == row["FirstNames"])
    ]

    if len(match) == 1:
        duplicates_in_matching.append(row)
    else:
        duplicates_not_in_matching.append(row)

duplicates_in_matching = pd.DataFrame(duplicates_in_matching)
duplicates_not_in_matching = pd.DataFrame(duplicates_not_in_matching)

In [ ]:
# merge the non duplicates by DoB
matches_non_duplicates_merged = pd.merge(matching, non_duplicates_in_matching, left_on="DoB", right_on="DoB_Formatted", how="inner")

In [ ]:
# merge the duplicates by DoB, Surname and Firstnames
matches_duplicates_merged = pd.merge(matching, duplicates_in_matching, left_on=["DoB", "SN", "FN"], right_on=["DoB_Formatted", "Surname", "FirstNames"], how="inner")

In [ ]:
matches_combined_merged = pd.concat([matches_non_duplicates_merged, matches_duplicates_merged])
matches_combined_merged.sort_values(by=["SN", "FN", "MeasurementDateTime"], inplace=True)
matches_combined_merged['Index_Matches'] = np.arange(0, len(matches_combined_merged))

In [ ]:
matches_combined_merged = matches_combined_merged.dropna(subset=['Label'])
matches_combined_merged["Label"] = matches_combined_merged["Label"].copy().astype(int)

In [ ]:
pat_ids_manual = set(matches_combined_merged["PatID"])

In [ ]:
rpw = pd.read_csv(f"{config.directory}/data/{config.rpw_data_csv}")
rpw["DateTime"] = pd.to_datetime(rpw["DateTime"], format='%Y-%m-%d %H:%M:%S', errors='coerce')

In [ ]:
folder = f"{config.directory}/data/rpw_numpy"
files = [f for f in os.listdir(folder) if f.endswith(".npy")]
session_counts = defaultdict(int)
for f in files:
    session_id = f.rsplit("_", 1)[0]
    session_counts[session_id] += 1
valid_sessions = {sid for sid, count in session_counts.items() if count >= 4}
rpw_filtered = rpw[rpw["SessionID"].isin(valid_sessions)].copy()

In [ ]:
merged = pd.merge(matches_combined_merged, rpw_filtered, left_on=["MeasurementDateTime"], right_on=["DateTime"], how="inner")

In [ ]:
pat_ids_merged = set(merged["PatID_x"])

In [ ]:
ecg_dates = pd.read_csv(f"{config.directory}/data/{config.ecg_dates_csv}")
ecg_dates['Date'] = pd.to_datetime(ecg_dates['Date'], format='%d.%m.%Y %H:%M:%S', errors='coerce')

In [ ]:
pairs = []

for pat_id, rpw_grp in merged.groupby("PatID_x"):
    ecg_grp = ecg_dates[ecg_dates["PatID"] == pat_id].copy()
    if ecg_grp.empty:
        continue

    used = set()

    for i, rpw_row in rpw_grp.iterrows():
        ecg_grp["diff"] = (ecg_grp["Date"] - rpw_row["MeasurementDateTime"]).abs()

        ecg_grp_filtered = ecg_grp[~ecg_grp.index.isin(used)]
        if ecg_grp_filtered.empty:
            continue

        nearest_idx = ecg_grp_filtered["diff"].idxmin()
        nearest_ecg = ecg_grp_filtered.loc[nearest_idx]

        if nearest_ecg["diff"].days > 0:
            continue

        pairs.append({
            "Fold": None,
            "Split": None,
            "PatID": pat_id,
            "SessionID": rpw_row["SessionID"],
            "ecg_file": nearest_ecg["Filename"],
            "Label": rpw_row["Label"],
            "rpw_time": rpw_row["MeasurementDateTime"],
            "ecg_time": nearest_ecg["Date"],
            "time_diff": nearest_ecg["diff"],
            "gender": rpw_row["GenderID"],
            "DoB": rpw_row["DoB_y"],
            "age": rpw_row["Alter"],
            "height": rpw_row["Größe"],
            "weight": rpw_row["Gewicht"],
            "bmi": rpw_row["BMI"] if pd.notna(obj=rpw_row["BMI"]) else (rpw_row["Gewicht"] / ((rpw_row["Größe"] / 100) ** 2)),
            "bsa": 0.007184 * (rpw_row["Größe"] ** 0.725) * (rpw_row["Gewicht"] ** 0.425),
            "train_years": rpw_row["Trainingsjahre"],
            "train_weekly": rpw_row["Training pro Woche"],
            "sbp_met_slope": rpw_row["SBP/MET slope2"],
            "sys_rest": rpw_row["RR sys"],
            "dia_rest": rpw_row["RR diast."],
            "sys_max": rpw_row["RR max sys"],
            "dia_max": rpw_row["RR max diast."],
            "sys_aortic": rpw_row["SysAortic"],
            "dia_aortic": rpw_row["DiaAortic"],
            "hr_rest": rpw_row["HF Ruhe EKG"],
            "hr_max": rpw_row["HF max. ERGO"],
            "echo": rpw_row["Schlagvolumen Echo"],
            "vitamin_d": rpw_row["25 OH Vitamin D"],
        })

        used.add(nearest_idx)

pairs_df = pd.DataFrame(pairs)

In [ ]:
sgkf_outer = StratifiedGroupKFold(n_splits=5)

split_set = 1
test_set = random.randint(1, 5)
for train_idx, test_idx in sgkf_outer.split(pairs_df, pairs_df['Label'], pairs_df['PatID']):
    if split_set == test_set:
        df_trainval = pairs_df.iloc[train_idx].copy()
        df_test = pairs_df.iloc[test_idx].copy()
        break
    split_set += 1

df_test["Split"] = "test"

In [ ]:
df_trainval["Split"] = "trainval"

In [ ]:
sgkf_inner = StratifiedGroupKFold(n_splits=config.num_folds)

folds = []
for fold_idx, (_, val_idx) in enumerate(sgkf_inner.split(df_trainval, df_trainval["Label"], df_trainval["PatID"]), start=1):
    fold_df = df_trainval.iloc[val_idx].copy()
    fold_df["Fold"] = fold_idx
    folds.append(fold_df)

df_folds = pd.concat(folds)
df_final = pd.concat([df_folds, df_test], ignore_index=True)
df_final

In [ ]:
df_final = df_final.sort_values("SessionID")
if not os.path.exists(f"{config.directory}/data"):
    os.makedirs(f"{config.directory}/data")
df_final.to_csv(f"{config.directory}/data/labels.csv", index=False)
generate_cohort_table(f"{config.directory}/data")